In [ ]:
# =====================================================================
# 🛸 第一步：一键安装视频扩散模型(DiT)所需的辅助核心库
# =====================================================================
!pip install av einops diffusers transformers accelerate peft --quiet

import importlib
import sys
if 'diffusers' in sys.modules:
    del sys.modules['diffusers']

import sys
import torch
import torch.nn as nn
import numpy as np
import av

In [ ]:
import diffusers
import transformers

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"当前代码正在使用: {device}")

print("\n" + "="*50)
print("🎉 恭喜！NanoDiT 视频模型开发环境已成功激活！")
print("="*50)
print(f"🐍 Python 版本  : {sys.version.split()[0]}")
print(f"🔥 PyTorch 版本 : {torch.__version__}")
print(f"⚡ GPU 算力支持 : {torch.cuda.is_available()} (可用显卡数量: {torch.cuda.device_count()})")

if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        props = torch.cuda.get_device_properties(i)
        print(f"   └─ 显卡 [{i}] 规格: {torch.cuda.get_device_name(i)} ({props.total_memory / 1024**3:.2f} GB 显存)")
else:
    print("   ⚠️ 警告: 目前处于 CPU 模式下。请在 Kaggle 右侧面板切换 Accelerator 为 GPU T4 x2 或 P100！")
print("="*50)

当前代码正在使用: cuda

🎉 恭喜！NanoDiT 视频模型开发环境已成功激活！
🐍 Python 版本  : 3.13.15
🔥 PyTorch 版本 : 2.11.0+cu128
⚡ GPU 算力支持 : True (可用显卡数量: 2)
   └─ 显卡 [0] 规格: Tesla T4 (14.56 GB 显存)
   └─ 显卡 [1] 规格: Tesla T4 (14.56 GB 显存)


In [8]:
# =====================================================================
# ⚙️ 核心前置：下载动漫视频数据集 + 流式挂载 5M 提示词数据集
# =====================================================================
import os
import glob
import kagglehub
from datasets import IterableDataset, load_dataset

print("🛰️ 正在并行连接 5M 提示词数据集与 314MB 视频数据集...")

# 下载本地视频（314MB）
ANIME_VIDEO_DIR = kagglehub.dataset_download("rrrkyawcrafter/anime-video")
print(f"✅ 视频数据集挂载成功！真实存储路径为: {ANIME_VIDEO_DIR}")

# 流式挂载 500 万提示词数据集（零本地占用）
hf_prompt_dataset = load_dataset("CaptionEmporium/anime-caption-danbooru-2021-sfw-5m-hq", streaming=True)
GLOBAL_HF_ITER = iter(hf_prompt_dataset['train'])

🛰️ 正在并行连接 5M 提示词数据集与 314MB 视频数据集...
✅ 视频数据集挂载成功！真实存储路径为: /kaggle/input/datasets/rrrkyawcrafter/anime-video


In [ ]:
import math
import os
import glob
import torch
import numpy as np
import av
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# =====================================================================
# 🎞️ 核心模块：手搓视频时空数据集 VideoDataset3D (多进程安全流式对齐版)
# =====================================================================
class VideoDataset3D(Dataset):
    def __init__(self, num_frames=16, height=128, width=128):
        """
        不再往里面传任何含有状态的 Dataset 或 Iterator 对象，确保此类纯净、100% 可被序列化！
        """
        self.num_frames = num_frames
        self.height = height
        self.width = width
        self.video_dir = video_dir

        # 1. 扫描所有视频文件路径（静态列表，完美支持 pickle）
        self.video_files = []
        for root, dirs, 

In [ ]:
files in os.walk(ANIME_VIDEO_DIR):  # 确保 ANIME_VIDEO_DIR 已定义
            for file in files:
                if file.lower().endswith('.mp4'):
                    self.video_files.append(os.path.join(root, file))

        print(f"📊 数据集扫描完毕：检测到 {len(self.video_files)} 个有效动漫视频文件。")
        
        # 2. 核心防御机制：在主进程不初始化流，声明一个占位符
        self.hf_iter = None
        self.last_worker_id = None

    def __len__(self):
        return len(self.video_files)

    def _init_worker_stream(self, worker_info):
        """
        🌟 绝杀招数：只有在子进程真正开始干活的时候，才在各自的子进程内存里初始化提示词流！
        各个进程互不干扰，完全绕过主进程到子进程的 pickle 打包传输。
        """
        from datasets import load_dataset
        # 这里请将 GLOBAL_HF_RAW 替换为你最原始加载该 Huggingface 数据集的那行代码
        # 比如：hf_dataset = load_dataset("你的数据集名字", streaming=True)
        # 这里假设全局变量 GLOBAL_HF_RAW 是你没用过 iter() 的流式数据集
        hf_dataset = GLOBAL_HF_ITER 
        self.hf_iter = iter(hf_dataset)
        
        # 🌟 核心对齐：为了让提示词和视频对齐，当前进程必须在提示词流中
        # 空转跳过前面进程领走的文本数量（每个子进程从各自起点同步前进）
        if worker_info is not None:
            per_worker = int(math.ceil(len(self.video_files) / float(worker_info.num_workers)))
            iter_start = worker_info.id * per_worker
            for _ in range(iter_start):
                try:
                    next(self.hf_iter)
                except StopIteration:
                    break

    def _read_video_av(self, video_path):
        # 💯 完整保留你原本最硬核的 PyAV 视频解码及预处理逻辑
        container = av.open(video_path)
        stream = container.streams.video
        frames = []
        for frame in container.decode(stream):
            img = frame.to_image().resize((self.width, self.height), Image.Resampling.BILINEAR)
            img_array = np.array(img, dtype=np.float32) / 255.0
            frames.append(img_array)
            if len(frames) == self.num_frames:
                break
        container.close()

        # 视频太短则复制最后一帧补满
        while len(frames) < self.num_frames:
            frames.append(frames[-1] if frames else np.zeros((self.height, self.width, 3), dtype=np.float32))

        video_np = np.stack(frames, axis=0)          # [T, H, W, C]
        return torch.from_numpy(video_np).permute(3, 0, 1, 2)  # [C, T, H, W]

    def __getitem__(self, idx):
        # 3. 检查当前处于哪个子进程
        worker_info = torch.utils.data.get_worker_info()
        current_worker_id = worker_info.id if worker_info is not None else -1
        
        # 4. 如果当前

In [ ]:
子进程还没初始化提示词流，或者进程切换了，立刻在子进程内部拉起流
        if self.hf_iter is None or current_worker_id != self.last_worker_id:
            self._init_worker_stream(worker_info)
            self.last_worker_id = current_worker_id

        # 5. 获取当前进程分配到的视频路径
        video_path = self.video_files[idx]

        # 6. 从当前进程独立的提示词流中同步抓取一行
        try:
            hf_sample = next(self.hf_iter)
            real_prompt = hf_sample.get("caption_llava_34b_no_tags_short", "1girl, anime style, high quality, colorful scene")
        except StopIteration:
            real_prompt = "1girl, anime style, high quality, colorful scene"
            
        if not real_prompt:
            real_prompt = "1girl, anime style, high quality, colorful scene"

        # 7. 解码视频
        try:
            video_tensor = self._read_video_av(video_path)
        except Exception as e:
            print(f"⚠️ 视频损坏读取失败: {video_path}, 错误: {e}")
            video_tensor = torch.zeros(3, self.num_frames, self.height, self.width)

        return video_tensor, real_prompt


# =====================================================================
# 🚀 第二步：现场测试与高效加载
# =====================================================================
if __name__ == "__main__":
    # 实例化数据集，类内部不持有外部生成器，100% 可以被 DataLoader 序列化打包
    video_dir

In [ ]:
=ANIME_VIDEO_DIR
    dataset = VideoDataset3D(video_dir,num_frames=4, height=128, width=128)
    
    if len(dataset) == 0:
        print("❌ 未找到视频，请检查 Kaggle 路径设置！")
    else:
        # 开启 num_workers=2 并行加速，此时绝对不可能再报 pickle 错误
        dataloader = DataLoader(
            dataset, 
            batch_size=1, 
            num_workers=0, 
            prefetch_factor=2,
            pin_memory=True
        )
        
        for videos, prompts in dataloader:
            print("🎉 视频张量形状 [B, C, T, H, W] :", videos.shape)
            print("📝 真实对齐的提示词:", prompts)
            break

TypeError: VideoDataset3D.__init__() got multiple values for argument 'num_frames'

In [ ]:
# =====================================================================
# 🛡️ 顶层补丁：修复 PreTrainedModel 丢失 + 隔离 torchao 量化冲突
# =====================================================================
import sys
import types
from importlib.machinery import ModuleSpec

import transformers as tf_mod
try:
    from transformers.modeling_utils import PreTrainedModel
except ImportError:
    from transformers.models.auto.modeling_auto import PreTrainedModel

sys.modules['transformers'].PreTrainedModel = PreTrainedModel
if not hasattr(tf_mod, 'PreTrainedModel'):
    setattr(tf_mod, 'PreTrainedModel', PreTrainedModel)

torchao_mock = types.ModuleType("torchao")
quantization_mock = types.ModuleType("torchao.quantization")
torchao_mock.__spec__ = ModuleSpec(name="torchao", loader=None, origin=None)
quantization_mock.__spec__ = ModuleSpec(name="torchao.quantization", loader=None, origin=None)
quantization_mock.FqnToConfig = None
quantization_mock.quantize_ = None
quantization_mock.DiffusersAutoQuantizer = None
sys.modules["torchao"] = torchao_mock
sys.modules["torchao.quantization"] = quantization_mock

print("🚀 [内存双重修补盾牌激活] PreTrainedModel 对齐 + torchao 隔离完成！")

# =====================================================================
# 🔄 加载原版大 VAE + T5 文本编码器
# =====================================================================
from diffusers import AutoencoderKL
from transformers import T5EncoderModel, T5Tokenizer

print("🛰️ 正在加载原版大 VAE 和 T5 文本编码器...")

vae = AutoencoderKL.from_pretrained("stabilityai/sd-vae-ft-mse", torch_dtype=torch.float16).cuda()
tokenizer = T5Tokenizer.from_pretrained("google/t5-v1_1-large")
text_encoder = T5EncoderModel.from_pretrained("google/t5-v1_1-large", torch_dtype=torch.float16).cuda()

vae.requires_grad_(False)
text_encoder.requires_grad_(False)
print("✅ VAE 与 T5 文本编码器已无阻碍，冷冻就位！")

In [ ]:
# =====================================================================
# 🔄 编码函数：真实视频 + 提示词 -> VAE 潜空间 + T5 语义向量
# =====================================================================
import os
import torch

def encode_video_and_text(video_tensor, prompt_list, video_dir):
    """
    video_tensor : [B, 3, 16, 64, 64] RGB 视频
    prompt_list  : 对应文本提示词列表
    video_dir    : 数据集根目录
    """
    exclude_files = ["Movie 4.mp4", "Movie.mp4", "Movie2.mp4", "Movie3.mp4"]
    video_files = [
        os.path.join(video_dir, f) for f in os.listdir(video_dir)
        if f.lower().endswith('.mp4') and f not in exclude_files
    ]

    B, C, T, H, W = video_tensor.shape

    # 视觉流：VAE 降维
    video_flattened = video_tensor.permute(0, 2, 1, 3, 4).flatten(0, 1).to(device="cuda", dtype=torch.float16)
    latents = vae.encode(video_flattened).latent_dist.sample() * 0.18215
    latents = latents.reshape(B, T, 4, H // 8, W // 8).permute(0, 2, 1, 3, 4)

    # 文本流：T5 语义
    text_inputs = tokenizer(prompt_list, padding="max_length", max_length=77, truncation=True, return_tensors="pt")
    text_input_ids = text_inputs.input_ids.to("cuda")
    text_features = text_encoder(text_input_ids).last_hidden_state

    return latents, text_features


# =====================================================================
# 现场测试
# =====================================================================
if __name__ == "__main__":
    video_dir = "/kaggle/input/datasets/rrrkyawcrafter/anime-video"
    mock_videos = torch.randn(2, 3, 16, 64, 64)          # ⚠️ 原代码末尾多了个 ]，已修复
    mock_prompts = ["1girl, anime style, high quality"]  # ⚠️ 原代码未定义，这里补上

    if len(mock_prompts) == 1 and mock_videos.shape[0] == 2:
        mock_prompts = mock_prompts * 2

    video_latents, text_features = encode_video_and_text(mock_videos, mock_prompts, video_dir)
    print("Video Latents 形状 :", video_latents.shape)   # [2, 4, 16, 8, 8]
    print("Text Features 形状 :", text_features.shape)   # [2, 77, 1024]

In [ ]:
# =====================================================================
# 🧱 Sora/Mini-DiT 骨架第一步：时空切块嵌入 VideoPatchEmbed
# =====================================================================
import torch
import torch.nn as nn

class VideoPatchEmbed(nn.Module):
    def __init__(
        self,
        num_frames=16,
        height=64,
        width=64,
        in_channels=4,
        patch_size=(2, 2, 2),
        hidden_size=512
    ):
        super().__init__()
        self.patch_size = patch_size
        self.proj = nn.Conv3d(
            in_channels=in_channels,
            out_channels=hidden_size,
            kernel_size=patch_size,
            stride=patch_size
        )
        self.num_patches_t = num_frames // patch_size[0]
        self.num_patches_h = height // patch_size[1]
        self.num_patches_w = width // patch_size[2]
        self.num_patches = self.num_patches_t * self.num_patches_h * self.num_patches_w

        print(f"🌲 VideoPatchEmbed 初始化成功！")
        print(f"   └─ Patch 大小: {patch_size}")
        print(f"   └─ Token 数量 L: {self.num_patches_t}×{self.num_patches_h}×{self.num_patches_w} = {self.num_patches}")
        print(f"   └─ 隐层维度 D: {hidden_size}")

    def forward(self, x):
        # [B, C, T, H, W] -> [B, D, T_p, H_p, W_p]
        x = self.proj(x)
        # -> [B, D, L]
        x = x.flatten(2)
        # -> [B, L, D]
        x = x.transpose(1, 2)
        return x


# =====================================================================
# 现场测试
# =====================================================================
if __name__ == "__main__":
    mock_video_latents = torch.randn(2, 4, 16, 8, 8)
    print("输入 Latents 形状:", mock_video_latents.shape)

    patch_embeder = VideoPatchEmbed(
        num_frames=16, height=8, width=8,
        in_channels=4, patch_size=(2, 2, 2), hidden_size=512
    )
    tokens = patch_embeder(mock_video_latents)
    print("输出 Tokens 形状:", tokens.shape)  # [2, 64, 512]

In [ ]:
# =====================================================================
# 🌀 Sora/Mini-DiT 骨架第二步：3D 旋转位置编码 (3D RoPE)
# =====================================================================
import torch
import torch.nn as nn

class MiniDiT3DRoPE(nn.Module):
    def __init__(self, hidden_size=512, t_dim=128, h_dim=192, w_dim=192):
        super().__init__()
        assert t_dim + h_dim + w_dim == hidden_size, "维度分配之和必须等于 hidden_size！"
        self.hidden_size = hidden_size
        self.t_dim = t_dim
        self.h_dim = h_dim
        self.w_dim = w_dim

    def _get_1d_rope(self, max_len, dim, device):
        inv_freq = 1.0 / (10000 ** (torch.arange(0, dim, 2).float().to(device) / dim))
        t = torch.arange(max_len, device=device, dtype=torch.float32)
        freqss = torch.outer(t, inv_freq)
        emb = torch.cat((freqss, freqss), dim=-1)
        return emb.cos(), emb.sin()

    def forward(self, x, num_patches_t, num_patches_h, num_patches_w):
        B, L, D = x.shape
        device = x.device

        cos_t, sin_t = self._get_1d_rope(num_patches_t, self.t_dim, device)
        cos_h, sin_h = self._get_1d_rope(num_patches_h, self.h_dim, device)
        cos_w, sin_w = self._get_1d_rope(num_patches_w, self.w_dim, device)

        cos_t = cos_t.view(num_patches_t, 1, 1, self.t_dim).expand(-1, num_patches_h, num_patches_w, -1)
        sin_t = sin_t.view(num_patches_t, 1, 1, self.t_dim).expand(-1, num_patches_h, num_patches_w, -1)
        cos_h = cos_h.view(1, num_patches_h, 1, self.h_dim).expand(num_patches_t, -1, num_patches_w, -1)
        sin_h = sin_h.view(1, num_patches_h, 1, self.h_dim).expand(num_patches_t, -1, num_patches_w, -1)
        cos_w = cos_w.view(1, 1, num_patches_w, self.w_dim).expand(num_patches_t, num_patches_h, -1, -1)
        sin_w = sin_w.view(1, 1, num_patches_w, self.w_dim).expand(num_patches_t, num_patches_h, -1, -1)

        cos_3d = torch.cat([cos_t, cos_h, cos_w], dim=-1).view(L, D)
        sin_3d = torch.cat([sin_t, sin_h, sin_w], dim=-1).view(L, D)

        x1 = x[..., :D // 2]
        x2 = x[..., D // 2:]
        rotate_x = torch.cat([-x2, x1], dim=-1)

        x_out = x * cos_3d.unsqueeze(0) + rotate_x * sin_3d.unsqueeze(0)
        return x_out


# =====================================================================
# 现场测试
# =====================================================================
if __name__ == "__main__":
    print("输入 Tokens 形状:", tokens.shape)
    rope_module = MiniDiT3DRoPE(hidden_size=512, t_dim=128, h_dim=192, w_dim=192).to(device)
    positioned_tokens = rope_module(
        tokens,
        num_patches_t=patch_embeder.num_patches_t,
        num_patches_h=patch_embeder.num_patches_h,
        num_patches_w=patch_embeder.num_patches_w
    )
    print("注入位置编码后的形状:", positioned_tokens.shape)

In [ ]:
# =====================================================================
# 🚨 GPU 安全防爆网
# =====================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

assert torch.cuda.is_available(), (
    "💥 危险！当前环境未开启 GPU 训练加速！\n"
    "防止 3D 时空注意力打爆 CPU 内存，已紧急拦截！"
)
device = torch.device("cuda")
print("✅ 显卡安全盾牌激活！当前运行设备:", torch.cuda.get_device_name(0))


# =====================================================================
# 🧠 MiniDiTBlock：AdaLN-Single + 时空自注意力 + 文本交叉注意力 + FFN
# =====================================================================
class MiniDiTBlock(nn.Module):
    def __init__(self, hidden_size=512, n_heads=8, d_context=1024):
        super().__init__()
        self.hidden_size = hidden_size
        self.n_heads = n_heads
        self.head_dim = hidden_size // n_heads

        self.adaln_ln = nn.Sequential(
            nn.SiLU(),
            nn.Linear(hidden_size, 6 * hidden_size, bias=True)
        )

        self.norm1 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.qkv = nn.Linear(hidden_size, hidden_size * 3, bias=True)
        self.proj = nn.Linear(hidden_size, hidden_size)

        self.norm2 = nn.LayerNorm(hidden_size, elementwise_affine=False, eps=1e-6)
        self.kv_context = nn.Linear(d_context, hidden_size * 2, bias=True)
        self.q_video = nn.Linear(hidden_size, hidden_size, bias=True)
        self.proj_context = nn.Linear(hidden_size, hidden_size)

        self.norm3 = nn.LayerNorm(hidden_size, elementwise_affine=True, eps=1e-6)
        self.ffn = nn.Sequential(
            nn.Linear(hidden_size, hidden_size * 4),
            nn.SiLU(),
            nn.Linear(hidden_size * 4, hidden_size)
        )  # ⚠️ 原代码这里缺了个右括号，已修复

        print(f"⚡ MiniDiTBlock 初始化成功！")

    def forward(self, x, t_embed, context=None):
        B, L, D = x.shape
        x = x.to(device)
        t_embed = t_embed.to(device)
        if context is not None:
            context = context.to(device)

        # AdaLN-Single 系数解包
        scale_msa, shift_msa, scale_scale_msa, scale_ca, shift_ca, scale_scale_ca = \
            self.adaln_ln(t_embed).chunk(6, dim=-1)

        # 时空联合自注意力
        x_norm = self.norm1(x) * (1 + scale_msa.unsqueeze(1)) + shift_msa.unsqueeze(1)
        qkv = self.qkv(x_norm).reshape(B, L, 3, self.n_heads, self.head_dim)
        q, k, v = qkv.chunk(3, dim=2)
        q = q.squeeze(2).permute(0, 2, 1, 3)
        k = k.squeeze(2).permute(0, 2, 1, 3)
        v = v.squeeze(2).permute(0, 2, 1, 3)
        attn_out = F.scaled_dot_product_attention(q, k, v)
        attn_out = attn_out.permute(0, 2, 1, 3).reshape(B, L, D)
        x = x + self.proj(attn_out) * scale_scale_msa.unsqueeze(1)

        # 文本交叉注意力
        if context is not None:
            L_ctx = context.shape[1]
            x_norm = self.norm2(x) * (1 + scale_ca.unsqueeze(1)) + shift_ca.unsqueeze(1)
            q_c = self.q_video(x_norm).reshape(B, L, self.n_heads, self.head_dim).permute(0, 2, 1, 3)
            kv_c = self.kv_context(context).reshape(B, L_ctx, 2, self.n_heads, self.head_dim)
            k_c, v_c = kv_c.chunk(2, dim=2)
            k_c = k_c.squeeze(2).permute(0, 2, 1, 3)
            v_c = v_c.squeeze(2).permute(0, 2, 1, 3)
            cross_out = F.scaled_dot_product_attention(q_c, k_c, v_c)
            cross_out = cross_out.permute(0, 2, 1, 3).reshape(B, L, D)
            x = x + self.proj_context(cross_out) * scale_scale_ca.unsqueeze(1)

        # FFN
        x = x + self.ffn(self.norm3(x))
        return x


# =====================================================================
# 🕐 TimestepEmbedder：时间标量 -> 向量
# =====================================================================
class TimestepEmbedder(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(1, hidden_size),
            nn.SiLU(),
            nn.Linear(hidden_size, hidden_size),
        )

    def forward(self, timesteps):
        return self.mlp(timesteps.unsqueeze(-1).float().to(device))


# =====================================================================
# 🛸 MiniDiTModel：完整去噪骨架
# =====================================================================
class MiniDiTModel(nn.Module):
    def __init__(self, num_frames=16, height=8, width=8, in_channels=4,
                 hidden_size=512, n_heads=8, d_context=1024, depth=4):
        super().__init__()
        self.num_patches_t = num_frames // 2
        self.num_patches_h = height // 2
        self.num_patches_w = width // 2

        self.patch_embed = VideoPatchEmbed(
            num_frames=num_frames, height=height, width=width,
            in_channels=in_channels, patch_size=(2, 2, 2), hidden_size=hidden_size
        ).to(device)

        self.time_embed = TimestepEmbedder(hidden_size).to(device)
        self.rope = MiniDiT3DRoPE(hidden_size=hidden_size).to(device)

        self.blocks = nn.ModuleList([
            MiniDiTBlock(hidden_size=hidden_size, n_heads=n_heads, d_context=d_context)
            for _ in range(depth)
        ]).to(device)

        out_channels = 2 * 2 * 2 * in_channels
        self.final_layer = nn.Sequential(
            nn.LayerNorm(hidden_size, eps=1e-6),
            nn.Linear(hidden_size, out_channels, bias=True)
        ).to(device)
        print(f"🎬 【MiniDiT 终极模型】深度={depth}，全模块驻留 GPU。")

    def forward(self, x, timesteps, context):
        x = x.to(device)
        timesteps = timesteps.to(device)
        context = context.to(device)

        x = self.patch_embed(x)
        t_emb = self.time_embed(timesteps)
        x = self.rope(x, self.num_patches_t, self.num_patches_h, self.num_patches_w)

        for block in self.blocks:
            x = block(x, t_embed=t_emb, context=context)

        return self.final_layer(x)


# =====================================================================
# 联动测试
# =====================================================================
if __name__ == "__main__":
    mock_noisy_latents = torch.randn(2, 4, 16, 8, 8, device=device)
    mock_timesteps = torch.tensor([50, 120], device=device)
    mock_contexts = torch.randn(2, 77, 1024, device=device)

    model = MiniDiTModel(num_frames=16, height=8, width=8, in_channels=4, depth=4)
    predicted_noise_patches = model(mock_noisy_latents, mock_timesteps, mock_contexts)
    print("预测噪声 Patch 形状:", predicted_noise_patches.shape)  # [2, 64, 32]

In [ ]:
# =====================================================================
# 🛸 训练准备：优化器 + AMP 缩放器
# =====================================================================
import torch.optim as optim
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = MiniDiTModel(num_frames=16, height=8, width=8, in_channels=4, depth=4).to(device)
optimizer = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')
print("🌱 优化器与混合精度护盾已就位！")


# =====================================================================
# 📉 多轮训练大循环
# =====================================================================
num_epochs = 3
steps_per_epoch = 4

train_dataset = VideoDataset3D( video_dir=ANIME_VIDEO_DIR,num_frames=16, height=64, width=64)
train_dataloader = DataLoader(train_dataset, batch_size=2, shuffle=True)

model.train()

for epoch in range(num_epochs):
    print(f"\n🎬 --- 开始第 【{epoch + 1}/{num_epochs}】 轮训练 ---")
    epoch_loss = 0.0
    data_iter = iter(train_dataloader)
    for step in range(steps_per_epoch):
        try:
            raw_videos, raw_prompts = next(data_iter)
        except StopIteration:
            data_iter = iter(train_dataloader)
            raw_videos, raw_prompts = next(data_iter)

        clean_videos, text_contexts = encode_video_and_text(
            raw_videos, list(raw_prompts))

        optimizer.zero_grad()

        with torch.amp.autocast('cuda'):
            # A. 随机噪声
            noise = torch.randn_like(clean_videos)
            # B. 随机时间步
            batch_size = clean_videos.shape[0]
            timesteps = torch.rand(batch_size, device=device)
            # C. Flow Matching 直线插值加噪
            t_expanded = timesteps.view(-1, 1, 1, 1, 1)
            noisy_videos = (1.0 - t_expanded) * clean_videos + t_expanded * noise
            # D. 前向预测速度
            target = noise - clean_videos
            predicted_velocity = model(noisy_videos, timesteps, text_contexts)
            # E. 目标按 patch 展开后计算 MSE
            pt, ph, pw = model.patch_embed.patch_size
            B_t, C_t, T_t, H_t, W_t = target.shape
            nt, nh, nw = T_t // pt, H_t // ph, W_t // pw
            target_patches = (
                target.reshape(B_t, C_t, nt, pt, nh, ph, nw, pw)
                      .permute(0, 2, 4, 6, 1, 3, 5, 7)
                      .reshape(B_t, nt * nh * nw, C_t * pt * ph * pw)
            )
            loss = F.mse_loss(predicted_velocity, target_patches)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        epoch_loss += loss.item()
        print(f"   🔹 Batch {step + 1}/{steps_per_epoch} | Loss: {loss.item():.6f}")

    avg_epoch_loss = epoch_loss / steps_per_epoch
    print(f"🏁 【第 {epoch + 1} 轮结束】 ➔ 平均 Loss: {avg_epoch_loss:.6f}")
    print("-" * 50)

print("\n🏆 训练闭环大获全胜！")

In [ ]:
# =====================================================================
# 🚀 MiniDiTPipeline：推理流水线（文本 -> 视频张量）
# =====================================================================
class MiniDiTPipeline:
    def __init__(self, model, vae, text_encoder, tokenizer, device="cuda"):
        self.model = model.eval()
        self.vae = vae.eval()
        self.text_encoder = text_encoder.eval()
        self.tokenizer = tokenizer
        self.device = device

    @torch.no_grad()
    def __call__(self, prompt, num_inference_steps=50, guidance_scale=1.0):
        model_dtype = next(self.model.parameters()).dtype

        with torch.amp.autocast('cuda'):
            # 1. 文本编码
            text_inputs = self.tokenizer([prompt], padding="max_length", max_length=77,
                                         truncation=True, return_tensors="pt")
            text_input_ids = text_inputs.input_ids.to(self.device)
            text_features = self.text_encoder(text_input_ids).last_hidden_state.to(dtype=model_dtype)

            # 2. 初始纯噪声
            latents = torch.randn(1, 4, 16, 8, 8, device=self.device, dtype=model_dtype)
            dt = 1.0 / num_inference_steps

            # 3. 欧拉积分去噪
            print(f"🎬 开启 {num_inference_steps} 步流匹配去噪...")
            for step in range(num_inference_steps):
                t_val = 1.0 - (step * dt)
                timesteps = torch.tensor([t_val], device=self.device, dtype=model_dtype)

                predicted_velocity = self.model(latents, timesteps, text_features)
                if len(predicted_velocity.shape) == 3:
                    velocity_field = predicted_velocity.view(1, 4, 16, 8, 8)
                else:
                    velocity_field = predicted_velocity

                latents = latents - velocity_field * dt
                if (step + 1) % 10 == 0 or step == 0:
                    print(f"   🔹 {step + 1}/{num_inference_steps} | t: {t_val:.2f}")

            # 4. VAE 解码（时空展平）
            print("🎉 去噪完成！正在 VAE 解码...")
            B, C, T, H, W = latents.shape
            latents_reshaped = latents.permute(0, 2, 1, 3, 4).flatten(0, 1)
            vae_dtype = next(self.vae.parameters()).dtype
            latents_reshaped = latents_reshaped.to(dtype=vae_dtype)
            decoded_images = self.vae.decode(latents_reshaped).sample
            video_frames = decoded_images.view(
                B, T, 3, decoded_images.shape[-2], decoded_images.shape[-1]
            ).permute(0, 2, 1, 3, 4)

        print("🏆 终极视频生成大获全胜！")
        return video_frames


# =====================================================================
# 组装流水线并生成
# =====================================================================
pipeline = MiniDiTPipeline(
    model=model,
    vae=vae,
    text_encoder=text_encoder,
    tokenizer=tokenizer,
    device=device
)

prompt_word = "1girl, anime style, high quality, golden hair, running in the rain"
generated_video = pipeline(prompt=prompt_word, num_inference_steps=30)
print(f"\n📊 最终视频张量形状: {generated_video.shape}")  # [1, 3, 16, 64, 64]

In [ ]:
# =====================================================================
# 🎞️ 视频渲染：5 维张量 -> 可播放 mp4 + Jupyter HTML5 播放器
# =====================================================================
!pip install imageio[ffmpeg] imageio-ffmpeg --quiet

import os
import numpy as np
import imageio
from IPython.display import HTML, display
from base64 import b64encode


def display_and_save_video(video_tensor, output_filename="generated_anime.mp4", fps=8):
    print("🎬 正在从显卡捞取张量，开始像素反归一化...")

    video = video_tensor[0]              # [C, T, H, W]
    video = video.permute(1, 2, 3, 0)    # [T, H, W, C]

    video = torch.clamp((video + 1.0) / 2.0, 0.0, 1.0)
    video_numpy = (video.cpu().numpy() * 255).astype(np.uint8)

    print(f"💾 正在压制视频... 目标: {output_filename}")
    writer = imageio.get_writer(output_filename, fps=fps, codec='libx264', pixelformat='yuv420p')
    for frame in video_numpy:
        writer.append_data(frame)
    writer.close()
    print("✅ 视频物理写盘成功！")

    mp4_file = open(output_filename, 'rb').read()
    data_url = "data:video/mp4;base64," + b64encode(mp4_file).decode()

    ui_html = f'''
    <div style="text-align: center; background: #1a1a1a; padding: 20px; border-radius: 10px; margin-top: 15px;">
        <h3 style="color: #00ffcc; font-family: sans-serif; margin-bottom: 10px;">🛸 Mini-DiT 梦境现实放映机</h3>
        <video width="320" height="320" controls autoplay loop style="border: 3px solid #00ffcc; border-radius: 8px;">
            <source src="{data_url}" type="video/mp4">
            您的浏览器不支持 HTML5 视频播放。
        </video>
        <p style="color: #aaa; font-size: 12px; margin-top: 8px;">提示：当前生成画面为未完全训练的混沌雪花或色块，代表推理链路 100% 成功！</p>
    </div>
    '''
    display(HTML(ui_html))


# =====================================================================
# 现场点火
# =====================================================================
display_and_save_video(generated_video, output_filename="my_first_sora_video.mp4", fps=8)

In [ ]:
import torch
import torch.optim as optim
import torch.nn.functional as F
import time

# 1. 设备与状态激活
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.train()  # 确保模型切换到训练模式

# 2. 优化器与混合精度护盾
optimizer = optim.AdamW(model.parameters(), lr=5e-4, weight_decay=0.01)
scaler = torch.amp.GradScaler('cuda')

num_epochs = 300  
start_time = time.time()

print("🚀 [VAE 潜空间实时压缩] 特训引擎已全开！正在将 3 通道视频压缩为 4 通道 Latents...")

for epoch in range(num_epochs):
    epoch_loss = 0.0
    batch_count = 0
    
    for step, batch_data in enumerate(dataloader): 
        # 解包你的真实多媒体数据流（此时 clean_videos 是原始 3 通道 RGB 视频）
        clean_videos_rgb, text_prompts = batch_data
        
        # 强行将原始视频张量驻留 GPU
        clean_videos_rgb = clean_videos_rgb.to(device)
        
        optimizer.zero_grad()
        
        # 🛡️ 混合精度护盾开启
        with torch.amp.autocast('cuda'):
            
            # =====================================================================
            # 🖼️ 核心修复 1：利用 VAE 编码器，将 [B, 3, 16, 64, 64] 压缩为 [B, 4, 16, 8, 8]
            # =====================================================================
            with torch.no_grad(): # 锁定 VAE 梯度，不需要训练它
                # 获取 VAE 接受的数据类型（通常是 Float32 或 Float16）
                vae_dtype = next(vae.parameters()).dtype
                inputs_for_vae = clean_videos_rgb.to(dtype=vae_dtype)
                
                # 💥 工业级跨维处理：2D VAE 无法处理 5 维视频，必须把 B 和 T 揉到一起伪装成图片
                B, C, T, H, W = inputs_for_vae.shape
                inputs_for_vae_reshaped = inputs_for_vae.permute(0, 2, 1, 3, 4).flatten(0, 1) # -> [B*T, 3, 64, 64]
                
                # 编码压缩
                posterior = vae.encode(inputs_for_vae_reshaped)
                # 采用最稳健的采样方式（或者直接取其均值 .mode()）
                latents_reshaped = posterior.latent_dist.sample() # -> [B*T, 4, 8, 8]
                
                # 逆向重塑：将压缩后的图片流重新拼回 5 维视频潜空间
                clean_videos = latents_reshaped.view(B, T, 4, 8, 8).permute(0, 2, 1, 3, 4) # -> [B, 4, 16, 8, 8]
                
                # 类型对齐到模型当前权重
                model_dtype = next(model.parameters()).dtype
                clean_videos = clean_videos.to(dtype=model_dtype)

            # =====================================================================
            # 🔤 核心修复 2：现场调用 T5 大模型，将文本转化为 1024 维张量
            # =====================================================================
            with torch.no_grad():
                text_inputs = tokenizer(
                    list(text_prompts), 
                    padding="max_length", 
                    max_length=77, 
                    truncation=True, 
                    return_tensors="pt"
                ).to(device)
                
                text_contexts = text_encoder(text_inputs.input_ids).last_hidden_state
                text_contexts = text_contexts.to(dtype=model_dtype)

            # =====================================================================
            # 🪐 标准流匹配流动训练逻辑（此时 clean_videos 已经是完美的 [B, 4, 16, 8, 8]）
            # =====================================================================
            # 【步骤 A】：生成与视频潜空间大小一致的纯高斯噪声
            noise = torch.randn_like(clean_videos)
            
            # 【步骤 B】：生成针对 Batch 大小的时间步 t
            timesteps = torch.rand(B, device=device, dtype=model_dtype)
            
            # 【步骤 C】：Flow Matching 直线流动加噪
            t_expanded = timesteps.view(-1, 1, 1, 1, 1)
            noisy_videos = (1.0 - t_expanded) * clean_videos + t_expanded * noise
            
            # 【步骤 D】：前向传播，模型在噪声中拼命学习猜测“去噪速度向量”
            target = noise - clean_videos
            predicted_velocity = model(noisy_videos, timesteps, text_contexts)
            
            # 【步骤 E】：切块并动态绑定特征投影对齐
            target_patches = model.patch_embed(target) 
            if not hasattr(model, "loss_align_layer"):
                model.loss_align_layer = torch.nn.Linear(target_patches.shape[-1], predicted_velocity.shape[-1]).to(device)
                
            target_patches_aligned = model.loss_align_layer(target_patches)
            loss = F.mse_loss(predicted_velocity, target_patches_aligned)
            
        # 💥 反向传播与梯度下降公式执行
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        epoch_loss += loss.item()
        batch_count += 1
        
    avg_epoch_loss = epoch_loss / max(1, batch_count)
    
    # 每 10 轮打印一次震撼的收敛进度
    if (epoch + 1) % 10 == 0 or epoch == 0:
        elapsed = (time.time() - start_time) / 60
        print(f"🎬 Epoch [{epoch + 1}/{num_epochs}] ➔ 记忆 Loss: {avg_epoch_loss:.6f} | 已训练: {elapsed:.2f} 分钟")

print(f"\n🏆 【方向 A 阶段通关】模型已成功过拟合！请立即重新运行你的 Pipeline 播放器代码！")